In [1]:
# IMPORTS
import os
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from glob import glob
import cv2
import numpy as np
from tqdm import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image

KeyboardInterrupt: 

In [ ]:

# -----------------------
# MODEL DEFINITIONS
# -----------------------


class Stem(nn.Module):
    def __init__(self, in_channels=1, out_channels=64, bias=False):
        super().__init__()
        self.conv = nn.Conv2d(in_channels,out_channels,kernel_size=7,stride=2,padding=3,bias=bias)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.max = nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

    def forward(self,x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.max(x)
        return x
    
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels,stride=1, bias=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=bias)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=bias)
        self.projectionConv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, padding=0, bias=bias)
        self.relu = nn.ReLU(inplace=True)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.bn1 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        skip = self.projectionConv(x)
        fx = self.conv1(x)
        fx = self.bn1(fx)
        fx = self.relu(fx)
        fx = self.conv2(fx)
        fx = self.bn2(fx)
        return F.relu(skip + fx)

class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(channels // r, 1))
        self.fc2 = nn.Linear(channels // r, channels)

    def forward(self, x):
        b, c, h, w = x.size()
        y = x.mean(dim=(2, 3))
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y))
        y = y.view(b, c, 1, 1)
        return x * y

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_blocks):
        super().__init__()

        # Initial conv
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        # Residual blocks (variable depth)
        self.res_blocks = nn.Sequential(
            *[ResBlock(out_channels, out_channels) for _ in range(num_blocks)]
        )

        # SE (SAX)
        self.se = SEBlock(out_channels)

        # Downsample
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.conv(x)
        x = self.res_blocks(x)
        x = self.se(x)
        skip = x            # save for decoder
        x = self.pool(x)    # downsample
        return x, skip
    

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)

        self.conv = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.res = ResBlock(out_channels, out_channels)
        self.se = SEBlock(out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x = self.res(x)
        x = self.se(x)
        return x


 
class ResSAXUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Stem
        self.stem = Stem(1, 64)

        # Encoder (3-4-6 structure)
        self.enc1 = EncoderBlock(64, 64, 3)
        self.enc2 = EncoderBlock(64, 128, 4)
        self.enc3 = EncoderBlock(128, 256, 6)

        # Deep layer
        self.enc4 = ResBlock(256, 512)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            ResBlock(512, 1024),
            SEBlock(1024)
        )

        # Decoder
        self.dec1 = DecoderBlock(1024, 256, 512)
        self.dec2 = DecoderBlock(512, 128, 256)
        self.dec3 = DecoderBlock(256, 64, 128)
        self.dec4 = DecoderBlock(128, 64, 64)

        # Final
        self.final = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):

        stem_out = self.stem(x)

        x, s1 = self.enc1(stem_out)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)

        x = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec1(x, s3)
        x = self.dec2(x, s2)
        x = self.dec3(x, s1)
        x = self.dec4(x, stem_out)  

        x = self.final(x)

        return x

In [ ]:
model = ResSAXUNet()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")